# 03 — SHAP Interpretability

**Author:** Virginia Galván, PhD

SHAP (SHapley Additive exPlanations) values are computed for the XGBoost model trained in Notebook 02, to identify which PAM50 genes drive each subtype prediction and to check those drivers against established breast-cancer subtype biology (ESR1, ERBB2, MKI67). A single test-set prediction is also explained individually, to show gene-level reasoning behind one specific case.

Designed to run in Google Colab or locally, after Notebooks 01 and 02.

In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split

IN_COLAB = "google.colab" in str(get_ipython())
DATA_DIR = "." if IN_COLAB else "../data"
FIGURES_DIR = "." if IN_COLAB else "../figures"
API_DIR = "." if IN_COLAB else "../api"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(API_DIR, exist_ok=True)

RANDOM_STATE = 42

**Required input:** `brca_pam50_dataset.csv` (Notebook 01) and `model.joblib` (Notebook 02). Run the cell below to upload both (Colab only — if running locally, place them in `../data/` and `../api/` respectively).

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Upload brca_pam50_dataset.csv and model.joblib:")
    files.upload()

## 1. Load model and reconstruct test set

The train/test split is reconstructed with the same random seed and parameters used in Notebook 02, so the test set explained here is identical to the one the model was evaluated on. Feature order is taken from the saved model bundle, not redefined, so it can never drift out of sync with what the model expects.

In [ ]:
model_bundle = joblib.load(os.path.join(API_DIR, "model.joblib"))
model = model_bundle["model"]
model_name = model_bundle["model_name"]
feature_order = model_bundle["feature_order"]
label_encoder = model_bundle["label_encoder"]
class_names = label_encoder.classes_

raw = pd.read_csv(os.path.join(DATA_DIR, "brca_pam50_dataset.csv"))
X = raw[feature_order]
y = label_encoder.transform(raw["SUBTYPE"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model_name, X_test.shape

## 2. Compute SHAP values

`TreeExplainer` computes exact SHAP values for tree-based models (no approximation needed, unlike model-agnostic SHAP methods). For a 5-class model, it returns one set of per-gene attributions per class — how much each gene pushed the prediction toward or away from that specific subtype.

In [ ]:
explainer = shap.TreeExplainer(model)
raw_shap = explainer.shap_values(X_test)

if isinstance(raw_shap, list):
    shap_values_per_class = raw_shap
else:
    shap_values_per_class = [raw_shap[:, :, i] for i in range(raw_shap.shape[2])]

len(shap_values_per_class), shap_values_per_class[0].shape

## 3. Global feature importance

For each gene, the mean absolute SHAP value is averaged across all 5 subtypes, giving a single overall importance ranking across the PAM50 panel.

In [ ]:
mean_abs_shap = np.mean(
    [np.abs(class_shap).mean(axis=0) for class_shap in shap_values_per_class],
    axis=0,
)

importance_df = pd.DataFrame({
    "gene": feature_order,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
importance_df["rank"] = importance_df.index + 1

top_n = 15
top_genes = importance_df.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top_genes["gene"], top_genes["mean_abs_shap"], color="#4EACD1")
ax.set_xlabel("Mean |SHAP value| (averaged across subtypes)")
ax.set_title(f"Top {top_n} genes by global SHAP importance — {model_name}")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig7_shap_global_importance.png"), dpi=150)
plt.show()

## 4. Biological cross-validation — known marker genes

ESR1, ERBB2, and MKI67 — the same three marker genes examined in Notebook 01's exploratory analysis — are checked against their SHAP importance ranking here. If a model has learned a biologically sound decision process, these established subtype markers should rank highly, not just correlate at the univariate level.

In [ ]:
marker_genes = ["ESR1", "ERBB2", "MKI67"]
importance_df[importance_df["gene"].isin(marker_genes)]

## 5. Explaining a single prediction

One test-set sample is explained individually: which genes, and by how much, pushed the model toward its predicted subtype for this specific case. Positive values push the prediction toward the predicted class; negative values push away from it.

In [ ]:
sample_idx = 0
sample_features = X_test.iloc[[sample_idx]]
predicted_class_idx = model.predict(sample_features)[0]
true_class = class_names[y_test[sample_idx]]
predicted_class = class_names[predicted_class_idx]

sample_shap = shap_values_per_class[predicted_class_idx][sample_idx]
sample_df = pd.DataFrame({
    "gene": feature_order,
    "shap_value": sample_shap,
}).sort_values("shap_value", key=np.abs, ascending=False).head(10).iloc[::-1]

colors = ["#c0392b" if v > 0 else "#2980b9" for v in sample_df["shap_value"]]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(sample_df["gene"], sample_df["shap_value"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"SHAP value (toward '{predicted_class}')")
ax.set_title(f"Sample {sample_idx} — true: {true_class}, predicted: {predicted_class}")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig8_shap_single_prediction.png"), dpi=150)
plt.show()